# Step 1 - Initial Dataset processing

```sql
create table "AllGalaxies"
(
    "OriginalDatasetId"             text,
    "DebiasedSpiralProbability"     float8,
    "DebiasedEllipticalProbability" float8,
    "OriginalDataset"                text,
    "ra"                            float8,
    "dec"                           float8,
    "Spec_z"                        float8,
    "Spec_z_err"                    float8,
    "Spec_mag"                      float8,
    "Spec_mag_err"                  float8
);
create unique index id_per_dataset
    on "AllGalaxies" ("OriginalDatasetId", "OriginalDataset");

create index original_id
    on "AllGalaxies" ("OriginalDatasetId");


## GZ1

In [1]:
import pandas as pd
from astropy.coordinates import SkyCoord
from astropy import units as unit

def map_ra(ra, dec):
    coords = SkyCoord(ra, dec, unit=(unit.deg, unit.deg)).transform_to("fk5")
    return coords.ra.deg

gz1_data = pd.read_csv("GZ1/GalaxyZoo1_DR_table2.csv")

# there is no data cleaning for GZ1 done
gz1_clean = gz1_data

gz1_clean["OriginalDataset"] = "GZ1"
gz1_clean = gz1_clean.rename(columns={
    "P_EL_DEBIASED" : "DebiasedEllipticalProbability",
    "P_CS_DEBIASED" : "DebiasedSpiralProbability",
    "OBJID": "OriginalDatasetId"
    }).reset_index(drop=True)

gz1_clean["DebiasedSpiralFlag"] = gz1_clean.apply(lambda r: 1 if r["SPIRAL"] == 1 else 0, axis = 1)
gz1_clean["DebiasedEllipticalFlag"] = gz1_clean.apply(lambda r: 1 if r["ELLIPTICAL"] == 1 else 0, axis = 1)

gz1_clean[["OriginalDatasetId", "DebiasedSpiralProbability", "DebiasedEllipticalProbability", "OriginalDataset"]].to_csv("GZ1/GZ1_unfiltered.csv", index=False)

### CasJobs query to get standard ra and dec

```sql
SELECT g.originaldatasetid, s.ra, s.dec
into mydb.GZ1_ALL_Coords
from MyDB.GZ1_ALL AS g
JOIN DR7.PhotoObj AS s
ON s.objID = g.originaldatasetid

### query to update ra and dec of GZ1 galaxies from results above

```sql
BEGIN TRANSACTION;

UPDATE "AllGalaxies" AS g
SET ra = c.ra, dec = c.dec
FROM "GZ1_uinfiltered_coords" AS c
WHERE g."OriginalDataset" = 'GZ1'
AND g."OriginalDatasetId" = c.originaldatasetid;

COMMIT TRANSACTION;

## GZ2

In [ ]:
gz2_data = pd.read_csv("GZ2/gz2_hart16.csv")

# remove all artefacts
gz2_clean = gz2_data[gz2_data["gz2_class"] != 'A']

gz2_clean["OriginalDataset"] = "GZ2"

gz2_clean = gz2_clean.rename(columns={
    "t01_smooth_or_features_a02_features_or_disk_flag" : "DebiasedSpiralFlag",
    "t01_smooth_or_features_a02_features_or_disk_debiased" : "DebiasedSpiralProbability",
    "t01_smooth_or_features_a01_smooth_flag" : "DebiasedEllipticalFlag",
    "t01_smooth_or_features_a01_smooth_debiased" : "DebiasedEllipticalProbability",
    "dr7objid": "OriginalDatasetId"
    }).reset_index(drop=True)

gz2_clean[["OriginalDatasetId", "DebiasedSpiralProbability", "DebiasedEllipticalProbability", "OriginalDataset", "ra", "dec"]].to_csv("GZ2/GZ2_unfiltered.csv", index= False)


## GZ Hubble

In [ ]:
gzh_data = pd.read_csv("GZH/gz_hubble_main.csv")

# remove all artefacts
gzh_clean = gzh_data[gzh_data["t01_smooth_or_features_a03_star_or_artifact_flag"] != 1]

gzh_clean["OriginalDataset"] = "GZ_HUBBLE"

gzh_clean = gzh_clean.rename(columns={
    "t01_smooth_or_features_a02_features_or_disk_flag" : "DebiasedSpiralFlag",
    "t01_smooth_or_features_a02_features_or_disk_debiased_fraction" : "DebiasedSpiralProbability",
    "t01_smooth_or_features_a01_smooth_flag" : "DebiasedEllipticalFlag",
    "t01_smooth_or_features_a01_smooth_debiased_fraction" : "DebiasedEllipticalProbability",
    "zooniverse_id": "OriginalDatasetId",
    "RA": "ra",
    "DEC": "dec"
    }).reset_index(drop=True)

gzh_clean[["OriginalDatasetId", "DebiasedSpiralProbability", "DebiasedEllipticalProbability", "OriginalDataset", "ra", "dec"]].to_csv("GZH/GZH_unfiltered.csv", index= False)

## GZ DECALS

In [ ]:
gzdec_1_2_data = pd.read_csv("GZD/gz_decals_volunteers_1_and_2.csv")

# # remove all artefacts
gzd_clean = gzdec_1_2_data[gzdec_1_2_data["smooth-or-featured_artifact_debiased"] < 0.5]

gzd_clean["OriginalDataset"] = "GZ_DECALS_12"

gzd_clean = gzd_clean.rename(columns={
    "smooth-or-featured_featured-or-disk_debiased" : "DebiasedSpiralProbability",
    "smooth-or-featured_smooth_debiased" : "DebiasedEllipticalProbability",
    "iauname": "OriginalDatasetId",
    }).reset_index(drop=True)

gzd_clean[["OriginalDatasetId", "DebiasedSpiralProbability", "DebiasedEllipticalProbability", "OriginalDataset", "ra", "dec"]].to_csv("GZD/GZD_12_unfiltered.csv", index= False)

In [ ]:
gzdec_5_data = pd.read_csv("GZD/gz_decals_volunteers_5.csv")

# # remove all artefacts
gzd_5_clean = gzdec_5_data[gzdec_5_data["smooth-or-featured_artifact_debiased"] < 0.5]

gzd_5_clean["OriginalDataset"] = "GZ_DECALS_5"

gzd_5_clean = gzd_5_clean.rename(columns={
    "smooth-or-featured_featured-or-disk_debiased" : "DebiasedSpiralProbability",
    "smooth-or-featured_smooth_debiased" : "DebiasedEllipticalProbability",
    "iauname": "OriginalDatasetId",
    }).reset_index(drop=True)

gzd_5_clean[["OriginalDatasetId", "DebiasedSpiralProbability", "DebiasedEllipticalProbability", "OriginalDataset", "ra", "dec"]].to_csv("GZD/GZD_5_unfiltered.csv", index= False)

## GZ CANDELS

In [ ]:
gzc_data = pd.read_csv("GZC/gz_candels_table_2_main_release.csv")

# # remove all artefacts
gzc_clean = gzc_data[gzc_data["t00_smooth_or_featured_a2_artifact_weighted_frac"] < 0.5]

gzc_clean["OriginalDataset"] = "GZ_CANDELS"

gzc_clean = gzc_clean.rename(columns={
    "t00_smooth_or_featured_a1_features_weighted_frac" : "DebiasedSpiralProbability",
    "t00_smooth_or_featured_a0_smooth_weighted_frac" : "DebiasedEllipticalProbability",
    "ID": "OriginalDatasetId",
    "RA": "ra",
    "Dec": "dec"
    }).reset_index(drop=True)

gzc_clean[["OriginalDatasetId", "DebiasedSpiralProbability", "DebiasedEllipticalProbability", "OriginalDataset", "ra", "dec"]].to_csv("GZC/GZC_unfiltered.csv", index= False)

---
# Step 2 - Data cleanup, analysis and merging by original dataset ID

## Assign a spiral certainty

```sql

BEGIN TRANSACTION;

ALTER TABLE "AllGalaxies"
ADD COLUMN "SpiralGalaxyCertainty" float8;

ALTER TABLE "AllGalaxies"
ADD COLUMN "DebiasedSpiralProbability_Clamped" float8;

ALTER TABLE "AllGalaxies"
ADD COLUMN "DebiasedEllipticalProbability_Clamped" float8;

UPDATE "AllGalaxies"
SET "DebiasedSpiralProbability_Clamped" = LEAST(GREATEST("DebiasedSpiralProbability", 0), 1)

UPDATE "AllGalaxies"
SET "DebiasedEllipticalProbability_Clamped" = LEAST(GREATEST("DebiasedEllipticalProbability", 0), 1)

UPDATE "AllGalaxies"
SET "SpiralGalaxyCertainty" =
        CASE 
             WHEN "DebiasedSpiralProbability_Clamped" = 0 THEN 0
             ELSE "DebiasedSpiralProbability_Clamped" / ("DebiasedSpiralProbability_Clamped" + "DebiasedEllipticalProbability_Clamped")
            END;

COMMIT TRANSACTION;

## Assign a certainty bucket

```sql
BEGIN TRANSACTION;

ALTER TABLE "AllGalaxies"
    ADD COLUMN "SpiralGalaxyCertaintyBucket" float8;

UPDATE "AllGalaxies"
SET "SpiralGalaxyCertaintyBucket" = width_bucket("SpiralGalaxyCertainty", array[0, 0.2, 0.4, 0.6, 0.8, 1.01]); -- 1.01 to include 1 in bucket #5

COMMIT TRANSACTION;
```

# Step 3 - Gather z and mag

## GZ1 and GZ2

Cas Job query
```postgresql
SELECT g.Id, g.OriginalDatasetId, s.z AS "Spec_z", s.zErr AS "Spec_z_err", sp.petroMag_r as "Spec_mag", sp.petroMagErr_r AS "Spec_mag_err"
FROM MyDB.MatchedGalaxiesById_GZ1_GZ2 AS g
  LEFT JOIN DR7.SpecObj AS s
  ON s.bestObjId = g.OriginalDatasetId
  AND s.objTypeName = 'GALAXY'
  LEFT JOIN DR7.SpecPhoto AS sp
  ON sp.objID = g.OriginalDatasetId
```

Import original csv file as GZ1_GZ2_z_and_mag_data table, preserving all values as float8
```postgresql
BEGIN TRANSACTION;

UPDATE "MatchedGalaxiesById" AS o
SET "Spec_z" = d."Spec_z", "Spec_z_err" = d."Spec_z_err", "Spec_mag" = d."Spec_mag", "Spec_mag_err" = d."Spec_mag_err"
FROM data."GZ1_GZ2_z_and_mag_data" AS d
WHERE d."Id" = o."Id"
AND d."Spec_z" IS NOT NULL;

COMMIT TRANSACTION;
```

## GZ Decals

Import original csv files as gz_decals_volunteers_1_and_2 and gz_decals_volunteers_5, preserving `redshift` as float8
```postgresql
BEGIN TRANSACTION;

UPDATE "MatchedGalaxiesById" AS o
SET "Spec_z" = d."redshift"
FROM data.gz_decals_volunteers_1_and_2 AS d
WHERE d."iauname" = o."OriginalDatasetId";

UPDATE "MatchedGalaxiesById" AS o
SET "Spec_z" = d."redshift"
FROM data.gz_decals_volunteers_5 AS d
WHERE d."iauname" = o."OriginalDatasetId";

COMMIT TRANSACTION;
```

## GZ Hubble

```postgresql
BEGIN TRANSACTION;

UPDATE "MatchedGalaxiesById" AS o
SET "Spec_z" = d."z_best"
FROM data.gz_hubble_main AS d
WHERE d."zooniverse_id" = o."OriginalDatasetId"
AND z_best != 999;

COMMIT TRANSACTION;
```

## GZ Candels

In [25]:
import pandas as pd
from astropy.coordinates import SkyCoord
from astropy import units as unit

# file source: https://archive.stsci.edu/hlsp/candels/cosmos-catalogs
candels_redshifts = pd.read_csv("GZC/hlsp_candels_hst_wfc3_cosmos_multi_v2_redshift-cat.txt", comment='#', sep=' ', header=None, usecols=range(1,8), names =[
"ID",
"RA",
"DEC",
"z_best",
"z_best_type",
"z_spec",
"z_spec_ref"
]).to_csv("GZC/gz_candels_cosmos_redshifts.csv", index=False)

```postgresql
BEGIN TRANSACTION;

UPDATE "MatchedGalaxiesById" AS o
SET "Spec_z" = d."z_best"
FROM data.gz_candels_cosmos_redshifts AS d
WHERE concat('COS_', d."id") = o."OriginalDatasetId";

ROLLBACK TRANSACTION;
```

In [29]:
import pandas as pd
from astropy.coordinates import SkyCoord
from astropy import units as unit

# file source: https://archive.stsci.edu/hlsp/candels/cosmos-catalogs
candels_redshifts = pd.read_csv("GZC/hlsp_candels_hst_wfc3_goodss_multi_v2_redshift-cat.txt", comment='#', sep=' ', header=None, usecols=range(1,8), names =[
"ID",
"RA",
"DEC",
"z_best",
"z_best_type",
"z_spec",
"z_spec_ref"
])

candels_redshifts.to_csv("GZC/gz_candels_gooods_redshifts.csv", index=False)

candels_redshifts

,ID,RA,DEC,z_best,z_best_type,z_spec,z_spec_ref
0,1,53.092996,-27.954538,0.5303,s,0.5303,VIMOS_10_
1,2,53.102888,-27.959662,1.7820,p,-1.0000,--
2,3,53.101686,-27.958479,0.2940,p,-1.0000,--
3,4,53.102280,-27.958651,0.6480,p,-1.0000,--
4,5,53.090559,-27.958499,2.7520,p,-1.0000,--
...,...,...,...,...,...,...,...
34925,34926,53.175520,-27.674899,2.8020,p,-1.0000,--
34926,34927,53.135252,-27.679915,0.4030,p,-1.0000,--
34927,34928,53.141342,-27.685999,3.9960,p,-1.0000,--
34928,34929,53.137204,-27.681651,2.2820,p,-1.0000,--


```postgresql
BEGIN TRANSACTION;

UPDATE "MatchedGalaxiesById" AS o
SET "Spec_z" = d."z_best"
FROM data.gz_candels_gooods_redshifts AS d
WHERE concat('GDS_', d."id") = o."OriginalDatasetId";

COMMIT TRANSACTION;

```

In [31]:
import pandas as pd
from astropy.coordinates import SkyCoord
from astropy import units as unit

# file source: https://archive.stsci.edu/hlsp/candels/cosmos-catalogs
candels_redshifts = pd.read_csv("GZC/hlsp_candels_hst_wfc3_uds_multi_v2_redshift-cat.txt", comment='#', sep=' ', header=None, usecols=range(1,8), names =[
"ID",
"RA",
"DEC",
"z_best",
"z_best_type",
"z_spec",
"z_spec_ref"
])

candels_redshifts.to_csv("GZC/gz_candels_uds_redshifts.csv", index=False)

candels_redshifts[candels_redshifts["ID"] == 1001]

,ID,RA,DEC,z_best,z_best_type,z_spec,z_spec_ref
1000,1001,34.490771,-5.271477,3.042,p,-1.0,--


```postgresql
BEGIN TRANSACTION;

UPDATE "MatchedGalaxiesById" AS o
SET "Spec_z" = d."z_best"
FROM data.gz_candels_uds_redshifts AS d
WHERE concat('UDS_', d."id") = o."OriginalDatasetId";

COMMIT TRANSACTION;
```

# Step 4 - Merge by coordinates

Add coordinates postgis column:

```sql
CREATE EXTENSION postgis;

SELECT *, st_point(ra, dec) AS "Coordinates"
INTO "MatchedGalaxiesWithCoordinates"
FROM "MatchedGalaxiesById";

-- spatial index
CREATE INDEX IDX_coordinates ON "MatchedGalaxiesWithCoordinates" USING GIST("Coordinates")
```


Create crossmatch columns:
```sql
BEGIN TRANSACTION;

ALTER TABLE "MatchedGalaxiesWithCoordinates"
    ADD COLUMN "Coordinates_GZ1_Id" text;

ALTER TABLE "MatchedGalaxiesWithCoordinates"
    ADD COLUMN "Coordinates_GZ2_Id" text;

ALTER TABLE "MatchedGalaxiesWithCoordinates"
    ADD COLUMN "Coordinates_GZD12_Id" text;

ALTER TABLE "MatchedGalaxiesWithCoordinates"
    ADD COLUMN "Coordinates_GZD5_Id" text;

ALTER TABLE "MatchedGalaxiesWithCoordinates"
    ADD COLUMN "Coordinates_GZH_Id" text;

ALTER TABLE "MatchedGalaxiesWithCoordinates"
    ADD COLUMN "Coordinates_GZC_Id" text;

COMMIT TRANSACTION;
```

## Cross-match by 0.1 arcsec distance
```sql
BEGIN TRANSACTION;

UPDATE "MatchedGalaxiesWithCoordinates"
SET "Coordinates_GZ1_Id" = null,
    "Coordinates_GZ2_Id" = null,
    "Coordinates_GZH_Id" = null,
    "Coordinates_GZC_Id" = null,
    "Coordinates_GZD12_Id" = null,
    "Coordinates_GZD5_Id" = null;

COMMIT TRANSACTION;

---------------------------GZ1------------------------------
BEGIN TRANSACTION;

WITH gz1 As (
    SELECT "OriginalDatasetId", "Coordinates", ra, dec, "Spec_z"
    FROM "MatchedGalaxiesWithCoordinates"
    WHERE "OriginalDataset" = 'GZ1'
),
sub AS (
    SELECT main."Id", main."OriginalDatasetId" AS "main_id", closest_gz1."OriginalDatasetId" AS "gz1_id", closest_gz1.dist
    FROM "MatchedGalaxiesWithCoordinates" AS main
    CROSS JOIN LATERAL (
        SELECT *, st_distance(gz1."Coordinates", main."Coordinates") AS dist
        FROM gz1
        -- within 0.1 arcsec
        WHERE st_dwithin(gz1."Coordinates", main."Coordinates", 0.000277778 / 10)
        -- prioritize same dataset matching (mostly for GZ1, GZ2 and GZD12, GZD5)
        ORDER BY main."OriginalDataset" = "OriginalDataset" DESC, main."OriginalDatasetId" = "OriginalDatasetId" DESC, dist ASC 
        LIMIT 1
    ) AS closest_gz1
)
UPDATE "MatchedGalaxiesWithCoordinates" AS main
SET "Coordinates_GZ1_Id" = sub."gz1_id"
FROM sub
WHERE main."Id" = sub."Id";

COMMIT TRANSACTION;

--------------------------GZ2-------------------------------

BEGIN TRANSACTION;

WITH gz2 As (
    SELECT "OriginalDatasetId", "Coordinates", ra, dec, "Spec_z"
    FROM "MatchedGalaxiesWithCoordinates"
    WHERE "OriginalDataset" = 'GZ2'
),
     sub AS (
         SELECT main."Id", main."OriginalDatasetId" AS "main_id", closest_gz2."OriginalDatasetId" AS "gz2_id", closest_gz2.dist
         FROM "MatchedGalaxiesWithCoordinates" AS main
                  CROSS JOIN LATERAL (
             SELECT *, st_distance(gz2."Coordinates", main."Coordinates") AS dist
             FROM gz2
             -- within 0.1 arcsec
             WHERE st_dwithin(gz2."Coordinates", main."Coordinates", 0.000277778 / 10)
             -- prioritize same dataset matching (mostly for GZ1, GZ2 and GZD12, GZD5)
             ORDER BY main."OriginalDataset" = "OriginalDataset" DESC, main."OriginalDatasetId" = "OriginalDatasetId" DESC, dist ASC
             LIMIT 1
             ) AS closest_gz2
     )
UPDATE "MatchedGalaxiesWithCoordinates" AS main
SET "Coordinates_GZ2_Id" = sub."gz2_id"
FROM sub
WHERE main."Id" = sub."Id";

COMMIT TRANSACTION;


--------------------------GZH-------------------------------

BEGIN TRANSACTION;

WITH gzh As (
    SELECT "OriginalDatasetId", "Coordinates", ra, dec, "Spec_z"
    FROM "MatchedGalaxiesWithCoordinates"
    WHERE "OriginalDataset" = 'GZ_HUBBLE'
),
     sub AS (
         SELECT main."Id", main."OriginalDatasetId" AS "main_id", closest_gzh."OriginalDatasetId" AS "gzh_id", closest_gzh.dist
         FROM "MatchedGalaxiesWithCoordinates" AS main
                  CROSS JOIN LATERAL (
             SELECT *, st_distance(gzh."Coordinates", main."Coordinates") AS dist
             FROM gzh
             -- within 0.1 arcsec
             WHERE st_dwithin(gzh."Coordinates", main."Coordinates", 0.000277778 / 10)
             -- prioritize same dataset matching (mostly for GZ1, GZ2 and GZD12, GZD5)
             ORDER BY main."OriginalDataset" = "OriginalDataset" DESC, main."OriginalDatasetId" = "OriginalDatasetId" DESC, dist ASC
             LIMIT 1
             ) AS closest_gzh
     )
UPDATE "MatchedGalaxiesWithCoordinates" AS main
SET "Coordinates_GZH_Id" = sub."gzh_id"
FROM sub
WHERE main."Id" = sub."Id";

COMMIT TRANSACTION;


--------------------------GZC-------------------------------
BEGIN TRANSACTION;

WITH gzc As (
    SELECT "OriginalDatasetId", "Coordinates", ra, dec, "Spec_z"
    FROM "MatchedGalaxiesWithCoordinates"
    WHERE "OriginalDataset" = 'GZ_CANDELS'
),
     sub AS (
         SELECT main."Id", main."OriginalDatasetId" AS "main_id", closest_gzc."OriginalDatasetId" AS "gzc_id", closest_gzc.dist
         FROM "MatchedGalaxiesWithCoordinates" AS main
                  CROSS JOIN LATERAL (
             SELECT *, st_distance(gzc."Coordinates", main."Coordinates") AS dist
             FROM gzc
             -- within 0.1 arcsec
             WHERE st_dwithin(gzc."Coordinates", main."Coordinates", 0.000277778 / 10)
             -- prioritize same dataset matching (mostly for GZ1, GZ2 and GZD12, GZD5)
             ORDER BY main."OriginalDataset" = "OriginalDataset" DESC, main."OriginalDatasetId" = "OriginalDatasetId" DESC, dist ASC
             LIMIT 1
             ) AS closest_gzc
     )
UPDATE "MatchedGalaxiesWithCoordinates" AS main
SET "Coordinates_GZC_Id" = sub."gzc_id"
FROM sub
WHERE main."Id" = sub."Id";

COMMIT TRANSACTION;


--------------------------GZD12-------------------------------

WITH gzd12 As (
    SELECT "OriginalDatasetId", "Coordinates", ra, dec, "Spec_z"
    FROM "MatchedGalaxiesWithCoordinates"
    WHERE "OriginalDataset" = 'GZ_DECALS_12'
),
     sub AS (
         SELECT main."Id", main."OriginalDatasetId" AS "main_id", closest_gzd12."OriginalDatasetId" AS "gzd12_id", closest_gzd12.dist
         FROM "MatchedGalaxiesWithCoordinates" AS main
                  CROSS JOIN LATERAL (
             SELECT *, st_distance(gzd12."Coordinates", main."Coordinates") AS dist
             FROM gzd12
             -- within 0.1 arcsec
             WHERE st_dwithin(gzd12."Coordinates", main."Coordinates", 0.000277778 / 10)
             -- prioritize same dataset matching (mostly for GZ1, GZ2 and GZD12, GZD5)
             ORDER BY main."OriginalDataset" = "OriginalDataset" DESC, main."OriginalDatasetId" = "OriginalDatasetId" DESC, dist ASC
             LIMIT 1
             ) AS closest_gzd12
     )
UPDATE "MatchedGalaxiesWithCoordinates" AS main
SET "Coordinates_GZD12_Id" = sub."gzd12_id"
FROM sub
WHERE main."Id" = sub."Id";

--------------------------GZD5-------------------------------

BEGIN TRANSACTION;

WITH gzd5 As (
    SELECT "OriginalDatasetId", "Coordinates", ra, dec, "Spec_z"
    FROM "MatchedGalaxiesWithCoordinates"
    WHERE "OriginalDataset" = 'GZ_DECALS_5'
),
     sub AS (
         SELECT main."Id", main."OriginalDatasetId" AS "main_id", closest_gzd5."OriginalDatasetId" AS "gzd5_id", closest_gzd5.dist
         FROM "MatchedGalaxiesWithCoordinates" AS main
                  CROSS JOIN LATERAL (
             SELECT *, st_distance(gzd5."Coordinates", main."Coordinates") AS dist
             FROM gzd5
             -- within 0.1 arcsec
             WHERE st_dwithin(gzd5."Coordinates", main."Coordinates", 0.000277778 / 10)
             -- prioritize same dataset matching (mostly for GZ1, GZ2 and GZD12, GZD5)
             ORDER BY main."OriginalDataset" = "OriginalDataset" DESC, main."OriginalDatasetId" = "OriginalDatasetId" DESC, dist ASC
             LIMIT 1
             ) AS closest_gzd5
     )
UPDATE "MatchedGalaxiesWithCoordinates" AS main
SET "Coordinates_GZD5_Id" = sub."gzd5_id"
FROM sub
WHERE main."Id" = sub."Id";

COMMIT TRANSACTION;
```

# Step 5 - Match by bin for each classification
-----
## (Repeat these steps for each bin count)

## Copy all data to class table

```sql
SELECT *
-- change class
INTO "FullyCrossMatchedGalaxies_5Class"
FROM "MatchedGalaxiesWithCoordinates";


ALTER TABLE "FullyCrossMatchedGalaxies_5Class"
ADD COLUMN "GZ1_5Class_Bin" INT,
ADD COLUMN "GZ2_5Class_Bin" INT,
ADD COLUMN "GZD12_5Class_Bin" INT,
ADD COLUMN "GZD5_5Class_Bin" INT,
ADD COLUMN "GZC_5Class_Bin" INT,
ADD COLUMN "GZH_5Class_Bin" INT;

UPDATE "FullyCrossMatchedGalaxies_5Class" AS main
SET "GZ1_5Class_Bin" = sub."SpiralGalaxyCertaintyBucket_5Class"
FROM "FullyCrossMatchedGalaxies_5Class" AS sub
WHERE sub."OriginalDataset" = 'GZ1'
    AND main."Coordinates_GZ1_Id" = sub."OriginalDatasetId";

UPDATE "FullyCrossMatchedGalaxies_5Class_New" AS main
SET "GZ2_5Class_Bin" = sub."SpiralGalaxyCertaintyBucket_5Class"
FROM "FullyCrossMatchedGalaxies_5Class_New" AS sub
WHERE sub."OriginalDataset" = 'GZ2'
    AND main."Coordinates_GZ2_Id" = sub."OriginalDatasetId";

UPDATE "FullyCrossMatchedGalaxies_5Class_New" AS main
SET "GZC_5Class_Bin" = sub."SpiralGalaxyCertaintyBucket_5Class"
FROM "FullyCrossMatchedGalaxies_5Class_New" AS sub
WHERE sub."OriginalDataset" = 'GZ_CANDELS'
  AND main."Coordinates_GZC_Id" = sub."OriginalDatasetId";

UPDATE "FullyCrossMatchedGalaxies_5Class_New" AS main
SET "GZH_5Class_Bin" = sub."SpiralGalaxyCertaintyBucket_5Class"
FROM "FullyCrossMatchedGalaxies_5Class_New" AS sub
WHERE sub."OriginalDataset" = 'GZ_HUBBLE'
  AND main."Coordinates_GZH_Id" = sub."OriginalDatasetId";

UPDATE "FullyCrossMatchedGalaxies_5Class_New" AS main
SET "GZD12_5Class_Bin" = sub."SpiralGalaxyCertaintyBucket_5Class"
FROM "FullyCrossMatchedGalaxies_5Class_New" AS sub
WHERE sub."OriginalDataset" = 'GZ_DECALS_12'
  AND main."Coordinates_GZD12_Id" = sub."OriginalDatasetId";

UPDATE "FullyCrossMatchedGalaxies_5Class_New" AS main
SET "GZD5_5Class_Bin" = sub."SpiralGalaxyCertaintyBucket_5Class"
FROM "FullyCrossMatchedGalaxies_5Class_New" AS sub
WHERE sub."OriginalDataset" = 'GZ_DECALS_5'
  AND main."Coordinates_GZD5_Id" = sub."OriginalDatasetId";
```

```sql

ALTER TABLE "FullyCrossMatchedGalaxies_5Class_New"
ADD COLUMN "AllBinsMatch" bool



UPDATE "FullyCrossMatchedGalaxies_5Class_New"
SET "AllBinsMatch" =
    CASE WHEN
    (
        WITH z as (SELECT DISTINCT UNNEST(array["GZ1_5Class_Bin", "GZ2_5Class_Bin", "GZC_5Class_Bin", "GZH_5Class_Bin", "GZD12_5Class_Bin", "GZD5_5Class_Bin"]) AS "Class")
        SELECT COUNT(*) FROM z
        WHERE z."Class" IS NOT NULL
    ) = 1
        THEN true
        ELSE false END;


DELETE FROM "FullyCrossMatchedGalaxies_5Class_New"
WHERE "AllBinsMatch" = false;
```

## Delete duplicate galaxies:
```sql
BEGIN TRANSACTION;

DELETE
FROM "FullyCrossMatchedGalaxies_5Class_New"
WHERE ctid IN (
    SELECT ctid
    FROM (
             SELECT ctid, "Coordinates_GZ1_Id", "Coordinates_GZ2_Id", "Coordinates_GZH_Id", "Coordinates_GZC_Id", "Coordinates_GZD12_Id", "Coordinates_GZD5_Id", row_number()
                   OVER (PARTITION BY "Coordinates_GZ1_Id", "Coordinates_GZ2_Id", "Coordinates_GZH_Id", "Coordinates_GZC_Id", "Coordinates_GZD12_Id", "Coordinates_GZD5_Id" ORDER BY ctid) rn
             FROM "FullyCrossMatchedGalaxies_5Class_New"
         ) t
    WHERE rn > 1
);

COMMIT TRANSACTION;
```

## Create Final dataset:
```sql
SELECT
    "Id",
    "OriginalDataset",
    "OriginalDatasetId",
    "Coordinates_GZ1_Id" AS "GZ1_Id",
    "Coordinates_GZ2_Id" AS "GZ2_Id",
    "Coordinates_GZH_Id" AS "GZH_Id",
    "Coordinates_GZC_Id" AS "GZC_Id",
    "Coordinates_GZD12_Id" AS "GZD12_Id",
    "Coordinates_GZD5_Id" AS "GZD5_Id",
    "ra",
    "dec",
    "Coordinates",
    "GZ1_5Class_Bin",
    "GZ2_5Class_Bin",
    "GZC_5Class_Bin",
    "GZH_5Class_Bin",
    "GZD12_5Class_Bin",
    "GZD5_5Class_Bin",
    "Spec_z",
    "Spec_z_err",
    "SpiralGalaxyCertainty",
    "SpiralGalaxyCertaintyBucket_5Class"
INTO "FinalDataset_5Class_New"
FROM "FullyCrossMatchedGalaxies_5Class_New"
```